# NeuroGolf Solver Family: Fill / Additive Marking

This notebook is a starter pipeline for the `fill_enclosed_regions` task family.

Workflow:

1. Load task ids from `task_groups/task_type_groups.json`.
2. Inspect the family metadata from `task_type_map.csv`.
3. Train or infer a per-task ONNX model with `train_family_task`.
4. Save `taskNNN.onnx` files.
5. Build `submission.zip` from the generated models.

The generated maps are heuristic solver-routing labels. Validate against visible examples before submitting.

In [1]:
# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None


BATCH, CH, H, W = 1, 10, 30, 30
MODEL_VERSION = "fill-additive-local3x3-small-v0.2"


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    candidates = [Path(path), Path("task_groups/task_type_map.csv")]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups_candidates = [Path(groups_path), Path("task_groups/task_type_groups.json")]
    for candidate in groups_candidates:
        if candidate.exists():
            groups = load_task_groups(candidate)
            return groups.get(family, [])
    raise FileNotFoundError(f"task_type_groups.json not found in: {groups_candidates}")


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None:
        raise ImportError("onnx is required to build models")


def make_model(nodes, initializers, opset=10):
    require_onnx()
    dt = TensorProto.FLOAT
    inp = helper.make_tensor_value_info("input", dt, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", dt, [BATCH, CH, H, W])
    graph = helper.make_graph(nodes, "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})


def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float32)
    bias = np.full((CH,), -0.5, dtype=np.float32)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = helper.make_tensor("W", dt, [CH, CH, 1, 1], weights.flatten().tolist())
    b = helper.make_tensor("B", dt, [CH], bias.tolist())
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    return {"right": right, "wrong": wrong, "first_wrong": first_wrong}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 4
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 4
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None


BATCH, CH, H, W = 1, 10, 30, 30
MODEL_VERSION = "fill-additive-local3x3-small-v0.2"


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    candidates = [Path(path), Path("task_groups/task_type_map.csv")]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups_candidates = [Path(groups_path), Path("task_groups/task_type_groups.json")]
    for candidate in groups_candidates:
        if candidate.exists():
            groups = load_task_groups(candidate)
            return groups.get(family, [])
    raise FileNotFoundError(f"task_type_groups.json not found in: {groups_candidates}")


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None:
        raise ImportError("onnx is required to build models")


def make_model(nodes, initializers, opset=10):
    require_onnx()
    dt = TensorProto.FLOAT
    inp = helper.make_tensor_value_info("input", dt, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", dt, [BATCH, CH, H, W])
    graph = helper.make_graph(nodes, "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})


def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float32)
    bias = np.full((CH,), -0.5, dtype=np.float32)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = helper.make_tensor("W", dt, [CH, CH, 1, 1], weights.flatten().tolist())
    b = helper.make_tensor("B", dt, [CH], bias.tolist())
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    return {"right": right, "wrong": wrong, "first_wrong": first_wrong}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 4
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 4
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path


def build_family_submission(family, trainer, data_dir, out_dir, fallback_identity=False, validate=False, task_ids_override=None):
    task_ids = list(task_ids_override) if task_ids_override is not None else family_task_ids(family)
    rows = []
    for task_id in task_ids:
        task = load_task(data_dir, task_id)
        model, info = trainer(task)
        if model is None and fallback_identity:
            model = make_identity_model()
            info = {**info, "fallback": "identity"}
        if model is None:
            rows.append({"task_id": task_id, "saved": False, **info})
            continue
        path = save_model(model, out_dir, task_id)
        row = {"task_id": task_id, "saved": True, "path": str(path), **info}
        if validate:
            try:
                row.update({f"visible_{k}": v for k, v in visible_validation_summary(path, task).items() if k != "first_wrong"})
            except Exception as exc:
                row["visible_error"] = repr(exc)
        rows.append(row)
    zip_path = create_submission_zip(out_dir)
    return rows, zip_path

In [2]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()

FAMILY = 'fill_enclosed_regions'
MODEL_VERSION = 'fill-additive-local3x3-task220-seed-ring-v0.7'
DATA_DIR, BASE_OUT_DIR = default_paths()
OUT_DIR = BASE_OUT_DIR / FAMILY
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)

DATA_DIR = /kaggle/input/competitions/neurogolf-2026
OUT_DIR = /kaggle/working/working_submission/fill_enclosed_regions
MODEL_VERSION = fill-additive-local3x3-task220-seed-ring-v0.7


## Fill / Additive Marking Version Contract

This notebook should track each solver version explicitly:

- `MODEL_VERSION`: human-readable solver version.
- selected tasks: rows from `task_type_map.csv` where `primary_family == "fill_enclosed_regions"`.
- architecture: ONNX nodes, ops, initializer shapes, file size, parameter count.
- performance: exact-match accuracy on `train`, `test`, `arc-gen`, and all visible examples.
- memory profile: parameter count, static tensor memory, runtime profile memory when `onnxruntime` is available.

The competition score for a correct task is driven by `params + memory_bytes`, so keep both visible.

In [3]:
# ONNX dependency setup for model export.
# Dry-run rule fitting can run without ONNX, but build_family_submission
# must import onnx to create taskNNN.onnx files.
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ['onnx', 'onnxruntime'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing ONNX packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import onnx
import onnxruntime as ort
from onnx import TensorProto, helper

print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing ONNX packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 18.0 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [4]:
task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()
local_3x3_df = family_df[family_df.candidate_flags.str.contains('local_3x3_consistent', na=False)].copy()
task_ids = local_3x3_df['task_id'].tolist()

print('family:', FAMILY)
print('family tasks:', len(family_df))
print('selected local_3x3 tasks:', len(task_ids))
display(local_3x3_df.head(5))

family: fill_enclosed_regions
family tasks: 59
selected local_3x3 tasks: 8


,task_id,task_num,primary_family,confidence,candidate_flags,n_train,n_test,n_arc_gen,n_examples,shape_relation,...,global_mapping,mapping_conflicts,fixed_geometric_transforms,local_3x3_score,local_3x3_conflicts,local_3x3_samples,input_nonzero_preserved_ratio,added_nonzero_cells,changed_cells,notes
14,task015,15,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,3,1,261,265,same_shape_variable_size,...,"{""0"":0,""1"":1,""2"":2,""6"":6,""8"":8}",1776,NaN,1.0,0,4860,1.0,1776,1776,Same shape; input is mostly preserved while ne...
80,task081,81,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,2,1,261,264,same_shape_variable_size,...,"{""0"":0,""8"":8}",634,NaN,1.0,0,2940,1.0,634,634,Same shape; input is mostly preserved while ne...
94,task095,95,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,2,1,262,265,same_shape_variable_size,...,"{""0"":0,""5"":5}",7368,NaN,1.0,0,4860,1.0,7368,7368,Same shape; input is mostly preserved while ne...
219,task220,220,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,4,1,262,267,same_shape_variable_size,...,"{""0"":0,""2"":2,""3"":3,""8"":8}",4248,NaN,1.0,0,9055,1.0,4248,4248,Same shape; input is mostly preserved while ne...
229,task230,230,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,3,1,262,266,same_shape_variable_size,...,"{""0"":0,""5"":5}",3392,NaN,1.0,0,10250,1.0,3392,3392,Same shape; input is mostly preserved while ne...


In [5]:
# Inspect one task quickly.
if task_ids:
    sample_task_id = task_ids[0]
    sample_task = load_task(DATA_DIR, sample_task_id)
    print(sample_task_id, 'examples:', len(all_examples(sample_task)))
    print('first input shape:', grid_shape(sample_task['train'][0]['input']))
    print('first output shape:', grid_shape(sample_task['train'][0]['output']))
    print('first input:', sample_task['train'][0]['input'])
    print('first output:', sample_task['train'][0]['output'])
else:
    print('No tasks currently mapped to this family.')

task015 examples: 265
first input shape: (9, 9)
first output shape: (9, 9)
first input: [[0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0]]
first output: [[0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 4, 0, 0, 0, 0, 0], [0, 0, 2, 0, 0, 0, 0, 0, 0], [0, 4, 0, 4, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 7, 0, 0], [0, 0, 0, 0, 0, 7, 1, 7, 0], [0, 0, 0, 0, 0, 0, 7, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0]]


In [6]:
# Local 3x3 selection table: these are the tasks this notebook version is responsible for.
selection_cols = [
    'task_id',
    'confidence',
    'n_train',
    'n_test',
    'n_arc_gen',
    'shape_relation',
    'input_shape_modes',
    'output_shape_modes',
    'input_color_list',
    'output_color_list',
    'new_output_color_list',
    'candidate_flags',
]
fill_selection = local_3x3_df[selection_cols].reset_index(drop=True)
print('selected local_3x3 fill/additive tasks:', len(fill_selection))
display(fill_selection)

selected local_3x3 fill/additive tasks: 8


,task_id,confidence,n_train,n_test,n_arc_gen,shape_relation,input_shape_modes,output_shape_modes,input_color_list,output_color_list,new_output_color_list,candidate_flags
0,task015,medium,3,1,261,same_shape_variable_size,9x9:265,9x9:265,"[0,1,2,6,8]","[0,1,2,4,6,7,8]","[4,7]",adds_new_color_preserves_input|local_3x3_consi...
1,task081,medium,2,1,261,same_shape_variable_size,7x7:264,7x7:264,"[0,8]","[0,1,8]",[1],adds_new_color_preserves_input|local_3x3_consi...
2,task095,medium,2,1,262,same_shape_variable_size,9x9:265,9x9:265,"[0,5]","[0,1,5]",[1],adds_new_color_preserves_input|local_3x3_consi...
3,task220,medium,4,1,262,same_shape_variable_size,15x15:37;14x14:36;13x13:31;12x12:29;11x11:29,15x15:37;14x14:36;13x13:31;12x12:29;11x11:29,"[0,2,3,8]","[0,1,2,3,4,6,8]","[1,4,6]",adds_new_color_preserves_input|local_3x3_consi...
4,task230,medium,3,1,262,same_shape_variable_size,15x15:149;10x10:117,15x15:149;10x10:117,"[0,5]","[0,1,2,3,4,5]","[1,2,3,4]",adds_new_color_preserves_input|local_3x3_consi...
5,task258,medium,3,1,262,same_shape_variable_size,10x10:52;7x7:49;9x9:45;6x6:43;8x8:39,10x10:52;7x7:49;9x9:45;6x6:43;8x8:39,"[0,1]","[0,1,2]",[2],adds_new_color_preserves_input|local_3x3_consi...
6,task331,medium,2,1,262,same_shape_variable_size,10x10:265,10x10:265,"[0,1]","[0,1,2,6,7,8]","[2,6,7,8]",adds_new_color_preserves_input|local_3x3_consi...
7,task352,medium,3,1,262,same_shape_variable_size,8x9:32;10x10:30;8x8:24;7x7:24;5x6:24,8x9:32;10x10:30;8x8:24;7x7:24;5x6:24,"[0,2,3,4,5,6,7,8,9]","[0,1,2,3,4,5,6,7,8,9]",[1],adds_new_color_preserves_input|local_3x3_consi...


In [7]:
# Identity-add fallback plus task220 seed-ring CNN trainer for local_3x3 fill/additive marking tasks.
#
# Architecture:
#   identity branch: Conv(1x1 input -> base logits)
#   correction branch: Conv(3x3 fixed/copy templates) -> Relu -> Conv(1x1 corrections)
#   output: Add(base logits, corrections)
#
# v6 keeps the v5 identity-add model, then tries a more general correction rule:
# if the output added at the center equals a stable neighboring input color, use
# a copy-neighbor wildcard template. The ONNX export expands that rule to one
# detector per possible copied nonzero color, which can be smaller and usually
# generalizes better than many exact color-specific templates.

CLEAR = 10
ZERO_HOT = -1
NO_CHANGE = -2
WILD = -99
MAX_TEMPLATE_DETECTORS = 2500
COPY_COLORS = tuple(range(1, 10))

# Keep center fixed so correction templates know which input channel to suppress.
REMOVAL_ORDER_3X3 = [0, 2, 6, 8, 1, 3, 5, 7]
REMOVAL_ORDERS_3X3 = [
    [0, 2, 6, 8, 1, 3, 5, 7],
    [1, 3, 5, 7, 0, 2, 6, 8],
    [0, 1, 2, 3, 5, 6, 7, 8],
    [8, 7, 6, 5, 3, 2, 1, 0],
    [1, 7, 3, 5, 0, 2, 6, 8],
    [0, 8, 2, 6, 1, 7, 3, 5],
]
OFFSETS_3X3 = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1), (0, 0), (0, 1),
    (1, -1), (1, 0), (1, 1),
]
CENTER_POS = 4
COPY_SOURCE_ORDER = [1, 3, 5, 7, 0, 2, 6, 8]


def input_canvas(grid):
    canvas = np.full((H, W), CLEAR, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def output_canvas(grid):
    canvas = np.full((H, W), ZERO_HOT, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def extract_patch_key(canvas, row, col, offsets=OFFSETS_3X3):
    vals = []
    for dr, dc in offsets:
        rr = row + dr
        cc = col + dc
        if 0 <= rr < H and 0 <= cc < W:
            vals.append(int(canvas[rr, cc]))
        else:
            vals.append(CLEAR)
    return tuple(vals)


def delta_color(input_color, output_color):
    if output_color == ZERO_HOT:
        return NO_CHANGE
    if input_color == output_color:
        return NO_CHANGE
    return int(output_color)


def template_matches(template, patch):
    return all(t == WILD or t == p for t, p in zip(template, patch))


def templates_overlap(a, b):
    for av, bv in zip(a, b):
        if av != WILD and bv != WILD and av != bv:
            return False
    return True


def template_specificity(template):
    return sum(1 for v in template if v != WILD)


def template_subsumes(general, specific):
    return all(g == WILD or g == s for g, s in zip(general, specific))


def learn_delta_rules(task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]

    patch_to_delta = {}
    conflicts = 0
    total_positions = 0
    changed_positions = 0

    for ex in examples:
        x = input_canvas(ex['input'])
        y = output_canvas(ex['output'])
        for r in range(H):
            for c in range(W):
                patch = extract_patch_key(x, r, c)
                color = delta_color(int(x[r, c]), int(y[r, c]))
                changed_positions += int(color != NO_CHANGE)
                total_positions += 1
                prev = patch_to_delta.get(patch)
                if prev is None:
                    patch_to_delta[patch] = color
                elif prev != color:
                    conflicts += 1

    return patch_to_delta, {
        'total_positions': total_positions,
        'changed_positions': changed_positions,
        'unique_patches': len(patch_to_delta),
        'conflicts': conflicts,
        'invalid_targets': 0,
    }


def fixed_rule_delta(rule, patch):
    return int(rule['delta'])


def copy_rule_delta(rule, patch):
    color = int(patch[int(rule['source_pos'])])
    if color in COPY_COLORS:
        return color
    return None


def rule_delta(rule, patch):
    if rule['type'] == 'copy':
        return copy_rule_delta(rule, patch)
    return fixed_rule_delta(rule, patch)


def rule_matches(rule, patch):
    if not template_matches(rule['template'], patch):
        return False
    return rule_delta(rule, patch) is not None


def rule_specificity(rule):
    return template_specificity(rule['template'])


def rule_detector_count(rule):
    return len(COPY_COLORS) if rule['type'] == 'copy' else 1


def is_rule_pure(rule, patch_to_delta):
    for patch, delta in patch_to_delta.items():
        if rule_matches(rule, patch) and rule_delta(rule, patch) != delta:
            return False
    return True


def patches_covered(rule, changed_patches):
    return {patch for patch in changed_patches if rule_matches(rule, patch) and rule_delta(rule, patch) == changed_patches[patch]}


def generalize_fixed_template_with_order(patch, target_delta, patch_to_delta, removal_order):
    template = list(patch)
    if template[CENTER_POS] != 0:
        return tuple(template)
    for _ in range(9):
        changed = False
        for pos in removal_order:
            if template[pos] == WILD:
                continue
            candidate = list(template)
            candidate[pos] = WILD
            rule = {'type': 'fixed', 'template': tuple(candidate), 'delta': int(target_delta)}
            if is_rule_pure(rule, patch_to_delta):
                template = candidate
                changed = True
        if not changed:
            break
    return tuple(template)


def generalize_fixed_template(patch, target_delta, patch_to_delta):
    candidates = [generalize_fixed_template_with_order(patch, target_delta, patch_to_delta, order) for order in REMOVAL_ORDERS_3X3]
    return sorted(candidates, key=lambda t: (template_specificity(t), t))[0]


def build_fixed_templates(patch_to_delta, changed_subset=None, blocked_rules=None):
    blocked_rules = blocked_rules or []
    if changed_subset is None:
        exact_changed = [(patch, delta) for patch, delta in patch_to_delta.items() if delta != NO_CHANGE]
    else:
        exact_changed = sorted(changed_subset.items(), key=lambda kv: (kv[1], kv[0]))

    accepted = []
    for patch, delta in sorted(exact_changed, key=lambda kv: (kv[1], kv[0])):
        template = generalize_fixed_template(patch, delta, patch_to_delta)
        rule = {'type': 'fixed', 'template': template, 'delta': int(delta)}
        if any(other['type'] == 'fixed' and other['delta'] != rule['delta'] and templates_overlap(rule['template'], other['template']) for other in accepted):
            rule = {'type': 'fixed', 'template': tuple(patch), 'delta': int(delta)}
        if any(rule_matches(blocked, patch) and rule_delta(blocked, patch) != delta for blocked in blocked_rules):
            rule = {'type': 'fixed', 'template': tuple(patch), 'delta': int(delta)}
        accepted.append(rule)

    unique = sorted({(r['template'], r['delta']) for r in accepted}, key=lambda kv: (kv[1], template_specificity(kv[0]), kv[0]))
    reduced = []
    for template, delta in unique:
        rule = {'type': 'fixed', 'template': template, 'delta': int(delta)}
        if any(other['delta'] == delta and template_subsumes(other['template'], template) for other in reduced):
            continue
        reduced.append(rule)
    return reduced


def generalize_copy_template_with_order(patch, source_pos, patch_to_delta, removal_order):
    template = list(patch)
    if template[CENTER_POS] != 0:
        return None
    if template[source_pos] not in COPY_COLORS:
        return None
    template[source_pos] = WILD
    for _ in range(9):
        changed = False
        for pos in removal_order:
            if pos in (CENTER_POS, source_pos) or template[pos] == WILD:
                continue
            candidate = list(template)
            candidate[pos] = WILD
            rule = {'type': 'copy', 'template': tuple(candidate), 'source_pos': int(source_pos)}
            if is_rule_pure(rule, patch_to_delta):
                template = candidate
                changed = True
        if not changed:
            break
    rule = {'type': 'copy', 'template': tuple(template), 'source_pos': int(source_pos)}
    return rule if is_rule_pure(rule, patch_to_delta) else None


def generalize_copy_template(patch, source_pos, patch_to_delta):
    candidates = []
    for order in REMOVAL_ORDERS_3X3:
        rule = generalize_copy_template_with_order(patch, source_pos, patch_to_delta, order)
        if rule is not None:
            candidates.append(rule)
    if not candidates:
        return None
    return sorted(candidates, key=lambda r: (rule_detector_count(r), rule_specificity(r), r['template']))[0]


def build_copy_candidates(patch_to_delta):
    changed = {patch: delta for patch, delta in patch_to_delta.items() if delta != NO_CHANGE}
    candidates = {}
    for patch, delta in changed.items():
        for source_pos in COPY_SOURCE_ORDER:
            if patch[source_pos] != delta:
                continue
            rule = generalize_copy_template(patch, source_pos, patch_to_delta)
            if rule is None:
                continue
            key = (rule['template'], rule['source_pos'])
            covered = patches_covered(rule, changed)
            if len(covered) < rule_detector_count(rule) + 1:
                continue
            previous = candidates.get(key)
            if previous is None or len(covered) > len(previous[1]):
                candidates[key] = (rule, covered)
    return list(candidates.values())


def build_copy_neighbor_rules(patch_to_delta):
    changed = {patch: delta for patch, delta in patch_to_delta.items() if delta != NO_CHANGE}
    fixed_baseline = build_fixed_templates(patch_to_delta)
    baseline_detectors = sum(rule_detector_count(r) for r in fixed_baseline)

    candidates = build_copy_candidates(patch_to_delta)
    candidates.sort(key=lambda item: (-(len(item[1]) - rule_detector_count(item[0])), rule_specificity(item[0]), item[0]['source_pos'], item[0]['template']))

    selected = []
    covered = set()
    for rule, candidate_coverage in candidates:
        new_coverage = candidate_coverage - covered
        if len(new_coverage) <= rule_detector_count(rule):
            continue
        selected.append(rule)
        covered.update(new_coverage)

    remaining = {patch: delta for patch, delta in changed.items() if patch not in covered}
    fixed_remaining = build_fixed_templates(patch_to_delta, remaining, blocked_rules=selected)
    candidate_rules = selected + fixed_remaining
    candidate_detectors = sum(rule_detector_count(r) for r in candidate_rules)

    if not selected or candidate_detectors > baseline_detectors:
        return fixed_baseline, {
            'copy_rule_count': 0,
            'fixed_rule_count': len(fixed_baseline),
            'expanded_detector_count': baseline_detectors,
            'baseline_detector_count': baseline_detectors,
            'detector_savings': 0,
        }

    return candidate_rules, {
        'copy_rule_count': len(selected),
        'fixed_rule_count': len(fixed_remaining),
        'expanded_detector_count': candidate_detectors,
        'baseline_detector_count': baseline_detectors,
        'detector_savings': baseline_detectors - candidate_detectors,
    }


def estimated_identity_add_params(detector_count):
    # base identity Conv W/B + detector Conv W/B + correction Conv W/B
    return int((CH * CH + CH) + detector_count * (CH * 3 * 3 + 1 + CH) + CH)


def fit_identity_add_rules(task):
    patch_to_delta, stats = learn_delta_rules(task)
    if stats['conflicts']:
        return None, {'ok': False, 'reason': 'delta patch conflicts', **stats}

    rules, rule_stats = build_copy_neighbor_rules(patch_to_delta)
    detector_count = sum(rule_detector_count(r) for r in rules)
    info = {
        'ok': False,
        'trainer': 'identity_add_copy_neighbor_3x3_cnn',
        'model_version': MODEL_VERSION,
        'kernel_shape': '[3, 3]',
        'pads': '[1, 1, 1, 1]',
        'max_detectors': MAX_TEMPLATE_DETECTORS,
        'template_count': detector_count,
        'rule_count': len(rules),
        'estimated_params': estimated_identity_add_params(detector_count),
        **rule_stats,
        **stats,
    }
    info['merged_template_count'] = stats['changed_positions'] - detector_count
    if detector_count > MAX_TEMPLATE_DETECTORS:
        return None, {**info, 'reason': 'too many template detectors'}
    if detector_count == 0:
        return None, {**info, 'reason': 'no added-cell templates'}

    return {'kind': 'identity_add_copy_neighbor', 'rules': rules}, {**info, 'ok': True, 'reason': None}


def expanded_detector_rules(payload):
    expanded = []
    for rule in payload['rules']:
        if rule['type'] == 'fixed':
            expanded.append((tuple(rule['template']), int(rule['delta'])))
            continue
        source_pos = int(rule['source_pos'])
        for color in COPY_COLORS:
            template = list(rule['template'])
            template[source_pos] = int(color)
            expanded.append((tuple(template), int(color)))
    return sorted(set(expanded), key=lambda kv: (kv[1], template_specificity(kv[0]), kv[0]))


def make_identity_add_model(payload):
    require_onnx()
    dt = TensorProto.FLOAT
    rules = expanded_detector_rules(payload)
    n_rules = len(rules)
    if n_rules == 0:
        return None

    W_base = np.zeros((CH, CH, 1, 1), dtype=np.float32)
    B_base = np.full((CH,), -0.5, dtype=np.float32)
    for c in range(CH):
        W_base[c, c, 0, 0] = 1.0

    W1 = np.zeros((n_rules, CH, 3, 3), dtype=np.float32)
    B1 = np.zeros((n_rules,), dtype=np.float32)
    for out_idx, (template, _delta) in enumerate(rules):
        ones = 0
        k = 0
        for kr in range(3):
            for kc in range(3):
                expected_color = template[k]
                if expected_color == WILD:
                    pass
                elif expected_color == CLEAR:
                    W1[out_idx, :, kr, kc] = -1.0
                else:
                    W1[out_idx, :, kr, kc] = -1.0
                    W1[out_idx, int(expected_color), kr, kc] = 1.0
                    ones += 1
                k += 1
        B1[out_idx] = -(ones - 0.5)

    W_delta = np.zeros((CH, n_rules, 1, 1), dtype=np.float32)
    B_delta = np.zeros((CH,), dtype=np.float32)
    for detector_idx, (template, delta) in enumerate(rules):
        center_color = template[CENTER_POS]
        if center_color == WILD or center_color == CLEAR:
            center_color = 0
        W_delta[int(delta), detector_idx, 0, 0] = 2.0
        W_delta[int(center_color), detector_idx, 0, 0] -= 2.0

    initializers = [
        helper.make_tensor('W_base', dt, list(W_base.shape), W_base.flatten().tolist()),
        helper.make_tensor('B_base', dt, list(B_base.shape), B_base.tolist()),
        helper.make_tensor('W_patch', dt, list(W1.shape), W1.flatten().tolist()),
        helper.make_tensor('B_patch', dt, list(B1.shape), B1.tolist()),
        helper.make_tensor('W_delta', dt, list(W_delta.shape), W_delta.flatten().tolist()),
        helper.make_tensor('B_delta', dt, list(B_delta.shape), B_delta.tolist()),
    ]
    nodes = [
        helper.make_node('Conv', ['input', 'W_base', 'B_base'], ['base_logits'], kernel_shape=[1, 1]),
        helper.make_node('Conv', ['input', 'W_patch', 'B_patch'], ['patch_logits'], kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node('Relu', ['patch_logits'], ['patch_hits']),
        helper.make_node('Conv', ['patch_hits', 'W_delta', 'B_delta'], ['delta_logits'], kernel_shape=[1, 1]),
        helper.make_node('Add', ['base_logits', 'delta_logits'], ['output']),
    ]
    return make_model(nodes, initializers, opset=10)



SEED_RING_OFFSETS = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1),
]


def fit_seed_ring_rules(task):
    """Fit a compact rule: every nonzero seed paints its 8-neighbor ring."""
    mapping = {}
    examples = all_examples(task)
    total_changed = 0
    seed_count = 0

    for ex in examples:
        x = input_canvas(ex['input'])
        y = output_canvas(ex['output'])
        seeds = [(r, c, int(x[r, c])) for r in range(H) for c in range(W) if 0 < int(x[r, c]) < CH]
        seed_count += len(seeds)
        predicted = np.array(x, copy=True)

        for sr, sc, seed_color in seeds:
            observed_ring_colors = set()
            for dr, dc in SEED_RING_OFFSETS:
                rr, cc = sr + dr, sc + dc
                if not (0 <= rr < H and 0 <= cc < W):
                    continue
                if int(x[rr, cc]) != 0:
                    continue
                out_color = int(y[rr, cc])
                if out_color != ZERO_HOT and out_color != 0:
                    observed_ring_colors.add(out_color)
            if not observed_ring_colors:
                continue
            if len(observed_ring_colors) != 1:
                return None, {'ok': False, 'trainer': 'seed_ring_color_remap_cnn', 'model_version': MODEL_VERSION, 'reason': 'seed maps to multiple ring colors'}
            ring_color = observed_ring_colors.pop()
            prev = mapping.get(seed_color)
            if prev is None:
                mapping[seed_color] = ring_color
            elif prev != ring_color:
                return None, {'ok': False, 'trainer': 'seed_ring_color_remap_cnn', 'model_version': MODEL_VERSION, 'reason': 'inconsistent seed color mapping'}

        for sr, sc, seed_color in seeds:
            ring_color = mapping.get(seed_color)
            if ring_color is None:
                continue
            for dr, dc in SEED_RING_OFFSETS:
                rr, cc = sr + dr, sc + dc
                if 0 <= rr < H and 0 <= cc < W and int(x[rr, cc]) == 0:
                    predicted[rr, cc] = ring_color

        expected = np.where(y == ZERO_HOT, x, y)
        total_changed += int(np.sum(predicted != x))
        if not np.array_equal(predicted, expected):
            return None, {
                'ok': False,
                'trainer': 'seed_ring_color_remap_cnn',
                'model_version': MODEL_VERSION,
                'reason': 'seed-ring rule does not explain visible examples',
                'seed_mapping': dict(sorted(mapping.items())),
                'seed_count': seed_count,
                'changed_positions': total_changed,
            }

    if not mapping:
        return None, {'ok': False, 'trainer': 'seed_ring_color_remap_cnn', 'model_version': MODEL_VERSION, 'reason': 'no seed-ring mapping learned'}

    return {'kind': 'seed_ring_color_remap', 'mapping': dict(sorted(mapping.items()))}, {
        'ok': True,
        'trainer': 'seed_ring_color_remap_cnn',
        'model_version': MODEL_VERSION,
        'reason': None,
        'seed_mapping': dict(sorted(mapping.items())),
        'seed_mapping_size': len(mapping),
        'seed_count': seed_count,
        'changed_positions': total_changed,
        'estimated_params': (CH * CH + CH) + (CH * CH * 3 * 3 + CH),
        'template_count': len(mapping),
        'expanded_detector_count': len(mapping),
    }


def make_seed_ring_model(payload):
    require_onnx()
    dt = TensorProto.FLOAT
    mapping = {int(k): int(v) for k, v in payload['mapping'].items()}

    W_base = np.zeros((CH, CH, 1, 1), dtype=np.float32)
    B_base = np.full((CH,), -0.5, dtype=np.float32)
    for c in range(CH):
        W_base[c, c, 0, 0] = 1.0

    W_ring = np.zeros((CH, CH, 3, 3), dtype=np.float32)
    B_ring = np.zeros((CH,), dtype=np.float32)
    for seed_color, ring_color in mapping.items():
        for dr, dc in SEED_RING_OFFSETS:
            kr, kc = dr + 1, dc + 1
            W_ring[ring_color, seed_color, kr, kc] += 2.0
            W_ring[0, seed_color, kr, kc] -= 2.0

    initializers = [
        helper.make_tensor('W_base', dt, list(W_base.shape), W_base.flatten().tolist()),
        helper.make_tensor('B_base', dt, list(B_base.shape), B_base.tolist()),
        helper.make_tensor('W_ring', dt, list(W_ring.shape), W_ring.flatten().tolist()),
        helper.make_tensor('B_ring', dt, list(B_ring.shape), B_ring.tolist()),
    ]
    nodes = [
        helper.make_node('Conv', ['input', 'W_base', 'B_base'], ['base_logits'], kernel_shape=[1, 1]),
        helper.make_node('Conv', ['input', 'W_ring', 'B_ring'], ['ring_logits'], kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node('Add', ['base_logits', 'ring_logits'], ['output']),
    ]
    return make_model(nodes, initializers, opset=10)


def train_family_task(task):
    seed_payload, seed_info = fit_seed_ring_rules(task)
    payload, info = fit_identity_add_rules(task)

    if seed_info.get('ok') and (not info.get('ok') or seed_info.get('estimated_params', 10**9) < info.get('estimated_params', 10**9)):
        model = make_seed_ring_model(seed_payload)
        if model is not None:
            return model, {'fallback_trainer': info.get('trainer'), 'fallback_estimated_params': info.get('estimated_params'), **seed_info}

    if not info.get('ok'):
        return None, {'seed_ring_reason': seed_info.get('reason'), **info}

    model = make_identity_add_model(payload)
    if model is None:
        return None, {'ok': False, 'reason': 'no identity-add copy-neighbor templates learned', **info}

    return model, {'seed_ring_reason': seed_info.get('reason'), **info}

In [8]:
# Dry-run seed-ring plus identity-add fallback fitting on the selected tasks without saving.
# This cell reports the exact selector decision. Actual model export still requires ONNX.
dry_rows = []
for task_id in task_ids[:10]:
    task = load_task(DATA_DIR, task_id)
    seed_payload, seed_info = fit_seed_ring_rules(task)
    fallback_payload, fallback_info = fit_identity_add_rules(task)
    use_seed_ring = bool(
        seed_info.get('ok') and (
            not fallback_info.get('ok') or
            seed_info.get('estimated_params', 10**9) < fallback_info.get('estimated_params', 10**9)
        )
    )
    payload = seed_payload if use_seed_ring else fallback_payload
    info = seed_info if use_seed_ring else {'seed_ring_reason': seed_info.get('reason'), **fallback_info}
    has_model = False
    export_error = None
    if info.get('ok') and onnx is not None:
        try:
            model = make_seed_ring_model(payload) if use_seed_ring else make_identity_add_model(payload)
            has_model = model is not None
        except Exception as exc:
            export_error = repr(exc)
    dry_rows.append({
        'task_id': task_id,
        'would_train': bool(info.get('ok')),
        'use_seed_ring': use_seed_ring,
        'seed_ok': bool(seed_info.get('ok')),
        'seed_estimated_params': seed_info.get('estimated_params'),
        'fallback_estimated_params': fallback_info.get('estimated_params'),
        'onnx_available': onnx is not None,
        'has_model': has_model,
        'export_error': export_error,
        **info,
    })

pd.DataFrame(dry_rows)

,task_id,would_train,use_seed_ring,seed_ok,seed_estimated_params,fallback_estimated_params,onnx_available,has_model,export_error,seed_ring_reason,...,total_positions,changed_positions,unique_patches,conflicts,invalid_targets,merged_template_count,reason,seed_mapping,seed_mapping_size,seed_count
0,task015,True,False,False,NaN,3554,True,True,None,seed-ring rule does not explain visible examples,...,238500.0,1776,458.0,0.0,0.0,1742.0,None,NaN,NaN,NaN
1,task081,True,False,False,NaN,524,True,True,None,seed-ring rule does not explain visible examples,...,237600.0,634,267.0,0.0,0.0,630.0,None,NaN,NaN,NaN
2,task095,True,False,True,1020.0,928,True,True,None,None,...,238500.0,7368,42.0,0.0,0.0,7360.0,None,NaN,NaN,NaN
3,task220,True,True,True,1020.0,5776,True,True,None,NaN,...,NaN,4248,NaN,NaN,NaN,NaN,None,"{2: 1, 3: 6, 8: 4}",3.0,531.0
4,task230,True,False,False,NaN,928,True,True,None,inconsistent seed color mapping,...,239400.0,3392,46.0,0.0,0.0,3384.0,None,NaN,NaN,NaN
5,task258,True,False,False,NaN,221,True,True,None,seed-ring rule does not explain visible examples,...,239400.0,1696,209.0,0.0,0.0,1695.0,None,NaN,NaN,NaN
6,task331,True,False,False,NaN,2039,True,True,None,seed maps to multiple ring colors,...,238500.0,4958,82.0,0.0,0.0,4939.0,None,NaN,NaN,NaN
7,task352,True,False,False,NaN,928,True,True,None,seed-ring rule does not explain visible examples,...,239400.0,3167,675.0,0.0,0.0,3159.0,None,NaN,NaN,NaN


In [9]:
# Build one model file for every selected local_3x3 task.
# This notebook version targets only tasks whose visible examples are consistent
# with a zero-conflict identity-add wildcard 3x3 model. No fallback models are emitted.
import shutil

# Clear stale models from earlier wider runs before creating this local_3x3 zip.
for old_model_path in OUT_DIR.glob('task*.onnx'):
    old_model_path.unlink()

rows, zip_path = build_family_submission(
    FAMILY,
    train_family_task,
    DATA_DIR,
    OUT_DIR,
    fallback_identity=False,
    validate=False,
    task_ids_override=task_ids,
)

result_df = pd.DataFrame(rows)
display(result_df)
saved_count = int(result_df.get('saved', pd.Series(dtype=bool)).sum()) if len(result_df) else 0
print('selected local_3x3 tasks:', len(task_ids))
print('models saved:', saved_count)
if len(result_df) and 'trainer' in result_df:
    display(result_df['trainer'].fillna('none').value_counts().rename_axis('trainer').reset_index(name='count'))

# Kaggle looks for /kaggle/working/submission.zip when submitting from a notebook.
submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
shutil.copy2(zip_path, submission_zip)
print('family zip:', zip_path)
print('kaggle submission zip:', submission_zip)

,task_id,saved,path,seed_ring_reason,ok,trainer,model_version,kernel_shape,pads,max_detectors,...,unique_patches,conflicts,invalid_targets,merged_template_count,reason,fallback_trainer,fallback_estimated_params,seed_mapping,seed_mapping_size,seed_count
0,task015,True,/kaggle/working/working_submission/fill_enclos...,seed-ring rule does not explain visible examples,True,identity_add_copy_neighbor_3x3_cnn,fill-additive-local3x3-task220-seed-ring-v0.7,"[3, 3]","[1, 1, 1, 1]",2500.0,...,458.0,0.0,0.0,1742.0,None,NaN,NaN,NaN,NaN,NaN
1,task081,True,/kaggle/working/working_submission/fill_enclos...,seed-ring rule does not explain visible examples,True,identity_add_copy_neighbor_3x3_cnn,fill-additive-local3x3-task220-seed-ring-v0.7,"[3, 3]","[1, 1, 1, 1]",2500.0,...,267.0,0.0,0.0,630.0,None,NaN,NaN,NaN,NaN,NaN
2,task095,True,/kaggle/working/working_submission/fill_enclos...,None,True,identity_add_copy_neighbor_3x3_cnn,fill-additive-local3x3-task220-seed-ring-v0.7,"[3, 3]","[1, 1, 1, 1]",2500.0,...,42.0,0.0,0.0,7360.0,None,NaN,NaN,NaN,NaN,NaN
3,task220,True,/kaggle/working/working_submission/fill_enclos...,NaN,True,seed_ring_color_remap_cnn,fill-additive-local3x3-task220-seed-ring-v0.7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,None,identity_add_copy_neighbor_3x3_cnn,5776.0,"{2: 1, 3: 6, 8: 4}",3.0,531.0
4,task230,True,/kaggle/working/working_submission/fill_enclos...,inconsistent seed color mapping,True,identity_add_copy_neighbor_3x3_cnn,fill-additive-local3x3-task220-seed-ring-v0.7,"[3, 3]","[1, 1, 1, 1]",2500.0,...,46.0,0.0,0.0,3384.0,None,NaN,NaN,NaN,NaN,NaN
5,task258,True,/kaggle/working/working_submission/fill_enclos...,seed-ring rule does not explain visible examples,True,identity_add_copy_neighbor_3x3_cnn,fill-additive-local3x3-task220-seed-ring-v0.7,"[3, 3]","[1, 1, 1, 1]",2500.0,...,209.0,0.0,0.0,1695.0,None,NaN,NaN,NaN,NaN,NaN
6,task331,True,/kaggle/working/working_submission/fill_enclos...,seed maps to multiple ring colors,True,identity_add_copy_neighbor_3x3_cnn,fill-additive-local3x3-task220-seed-ring-v0.7,"[3, 3]","[1, 1, 1, 1]",2500.0,...,82.0,0.0,0.0,4939.0,None,NaN,NaN,NaN,NaN,NaN
7,task352,True,/kaggle/working/working_submission/fill_enclos...,seed-ring rule does not explain visible examples,True,identity_add_copy_neighbor_3x3_cnn,fill-additive-local3x3-task220-seed-ring-v0.7,"[3, 3]","[1, 1, 1, 1]",2500.0,...,675.0,0.0,0.0,3159.0,None,NaN,NaN,NaN,NaN,NaN


selected local_3x3 tasks: 8
models saved: 8


,trainer,count
0,identity_add_copy_neighbor_3x3_cnn,7
1,seed_ring_color_remap_cnn,1


family zip: /kaggle/working/working_submission/fill_enclosed_regions/submission.zip
kaggle submission zip: /kaggle/working/submission.zip


In [10]:
# Model/version manifest for this notebook run.
run_manifest = {
    'family': FAMILY,
    'model_version': MODEL_VERSION,
    'task_count': len(task_ids),
    'out_dir': str(OUT_DIR),
}
run_manifest

{'family': 'fill_enclosed_regions',
 'model_version': 'fill-additive-local3x3-task220-seed-ring-v0.7',
 'task_count': 8,
 'out_dir': '/kaggle/working/working_submission/fill_enclosed_regions'}

In [11]:
# Optional: validate saved ONNX models on visible examples.
# This can be slow for large families and requires onnxruntime.
validate_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    summary = visible_validation_summary(row['path'], task)
    validate_rows.append({
        'task_id': row['task_id'],
        'right': summary['right'],
        'wrong': summary['wrong'],
    })

pd.DataFrame(validate_rows)

,task_id,right,wrong
0,task015,265,0
1,task081,264,0
2,task095,265,0
3,task220,267,0
4,task230,266,0
5,task258,266,0
6,task331,265,0
7,task352,266,0


In [12]:
# Submission helper.
# For a full competition submission, combine models from multiple family
# folders into one directory, then call create_submission_zip(combined_dir).
submission_zip = create_submission_zip(OUT_DIR)
print(submission_zip)

/kaggle/working/working_submission/fill_enclosed_regions/submission.zip


In [13]:
# Architecture, performance, and memory report for saved models.
# This cell expects train_family_task to save one or more ONNX models.
# It reports the metrics the competition cares about: file size, parameter
# count, and memory profile, plus train/test/arc-gen exact-match performance.

report_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    try:
        report = model_report(row['path'], task=task)
        arch = report['architecture']
        mem = report['memory_profile']
        perf = report['performance']
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'file_size_bytes': arch.get('file_size_bytes'),
            'params': arch.get('params'),
            'nodes': arch.get('nodes'),
            'op_counts': json.dumps(arch.get('op_counts', {}), sort_keys=True),
            'static_memory_bytes': mem.get('static_memory_bytes'),
            'runtime_memory_bytes': mem.get('runtime_memory_bytes'),
            'train_right': perf['train']['right'],
            'train_total': perf['train']['total'],
            'train_accuracy': perf['train']['accuracy'],
            'test_right': perf['test']['right'],
            'test_total': perf['test']['total'],
            'test_accuracy': perf['test']['accuracy'],
            'arc_gen_right': perf['arc_gen']['right'],
            'arc_gen_total': perf['arc_gen']['total'],
            'arc_gen_accuracy': perf['arc_gen']['accuracy'],
            'visible_right': perf['visible_all']['right'],
            'visible_total': perf['visible_all']['total'],
            'visible_accuracy': perf['visible_all']['accuracy'],
        })
    except Exception as exc:
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'profile_error': repr(exc),
        })

profile_df = pd.DataFrame(report_rows)
display(profile_df)

,task_id,model_version,file_size_bytes,params,nodes,op_counts,static_memory_bytes,runtime_memory_bytes,train_right,train_total,train_accuracy,test_right,test_total,test_accuracy,arc_gen_right,arc_gen_total,arc_gen_accuracy,visible_right,visible_total,visible_accuracy
0,task015,fill-additive-local3x3-task220-seed-ring-v0.7,14734,3554,5,"{""Add"": 1, ""Conv"": 3, ""Relu"": 1}",316800,352800,3,3,1.0,1,1,1.0,261,261,1.0,265,265,1.0
1,task081,fill-additive-local3x3-task220-seed-ring-v0.7,2612,524,5,"{""Add"": 1, ""Conv"": 3, ""Relu"": 1}",100800,136800,2,2,1.0,1,1,1.0,261,261,1.0,264,264,1.0
2,task095,fill-additive-local3x3-task220-seed-ring-v0.7,4228,928,5,"{""Add"": 1, ""Conv"": 3, ""Relu"": 1}",129600,165600,2,2,1.0,1,1,1.0,262,262,1.0,265,265,1.0
3,task220,fill-additive-local3x3-task220-seed-ring-v0.7,4439,1020,3,"{""Add"": 1, ""Conv"": 2}",72000,108000,4,4,1.0,1,1,1.0,262,262,1.0,267,267,1.0
4,task230,fill-additive-local3x3-task220-seed-ring-v0.7,4228,928,5,"{""Add"": 1, ""Conv"": 3, ""Relu"": 1}",129600,165600,3,3,1.0,1,1,1.0,262,262,1.0,266,266,1.0
5,task258,fill-additive-local3x3-task220-seed-ring-v0.7,1398,221,5,"{""Add"": 1, ""Conv"": 3, ""Relu"": 1}",79200,115200,3,3,1.0,1,1,1.0,262,262,1.0,266,266,1.0
6,task331,fill-additive-local3x3-task220-seed-ring-v0.7,8672,2039,5,"{""Add"": 1, ""Conv"": 3, ""Relu"": 1}",208800,244800,2,2,1.0,1,1,1.0,262,262,1.0,265,265,1.0
7,task352,fill-additive-local3x3-task220-seed-ring-v0.7,4228,928,5,"{""Add"": 1, ""Conv"": 3, ""Relu"": 1}",129600,165600,3,3,1.0,1,1,1.0,262,262,1.0,266,266,1.0


In [14]:
# Persist run metadata next to the generated models.
if 'profile_df' in globals() and len(profile_df):
    profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
    profile_df.to_csv(profile_path, index=False)
    print('wrote profile:', profile_path)

manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote manifest:', manifest_path)

wrote profile: /kaggle/working/working_submission/fill_enclosed_regions/fill_enclosed_regions_fill-additive-local3x3-task220-seed-ring-v0.7_profile.csv
wrote manifest: /kaggle/working/working_submission/fill_enclosed_regions/fill_enclosed_regions_fill-additive-local3x3-task220-seed-ring-v0.7_manifest.json
